*Módulo 4 de 9*

> **Prefer English?** Open [`04_vegetation_indices.ipynb`](../en/04_vegetation_indices.ipynb) — it is the same module, in English.


# 🌱 Módulo 4 — Índices de vegetación

🧭 **Objetivos** — convertir bandas crudas en significado. Calcular tú mismo
el **NDVI** a partir de las bandas roja y NIR, verificarlo contra la capa ya
calculada del tile, y aprender qué aportan los otros 6 índices del tile.
Entender por qué toda una *familia* de índices — no solo NDVI — ayuda a un
clasificador a distinguir cultivos.

📚 **La idea.** En el Módulo 2 viste la firma espectral de un cultivo saltar
en el infrarrojo cercano mientras el rojo se queda bajo. Un **índice**
destila ese contraste en un solo número. El más famoso es el **NDVI**:

$$ NDVI = \frac{NIR - Rojo}{NIR + Rojo} $$

Va de −1 (agua), pasa por ~0 (suelo desnudo), hasta ~+0.9 (cultivo denso y
sano). Por ser un *cociente*, cancela diferencias de brillo (ángulo solar,
pendiente) y lee el vigor de la planta directamente.

📚 **¿Por qué más que NDVI?** El NDVI se satura en doseles muy densos y no
dice nada del agua ni del residuo del suelo. Por eso el tile también trae
**EVI** y **GCVI** (clorofila), **MSAVI2** (ajustado al suelo), **LSWI**
(agua), **NDSVI** y **NDTI** (residuo / labranza). Juntos, 6 bandas + 7
índices = **13 capas** que describen cada píxel — más pistas para el
clasificador.

![bandas y NDVI](../../anim/es/04_bands_ndvi.svg)


In [ ]:
# Trae el tile del taller (pocos MB; queda en caché tras la primera descarga)
import os, sys

async def trae_archivo(name):
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/abxda/portable-geocrop/main/files/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url)
            open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request
            urllib.request.urlretrieve(url, dest)
    return dest

TILE = await trae_archivo("crop_tile_384.tif")
print("Tile listo:", TILE)

## Calcula el NDVI tú mismo, luego verifícalo

El tile guarda enteros escalados por 10000. El rojo es la banda índice 2, el
NIR es la banda índice 3. Calcula el NDVI con la fórmula — una línea NumPy
vectorizada, como en el Módulo 1 — y luego compáralo con la capa 6, que el
pipeline ya calculó. Deben coincidir.


In [ ]:
import numpy as np, rasterio, matplotlib.pyplot as plt

with rasterio.open(TILE) as src:
    img = src.read().astype(np.float64)

rojo = img[2]
nir = img[3]
ndvi_mio  = (nir - rojo) / (nir + rojo + 1e-9)  # +minúsculo para no dividir entre 0
ndvi_tile = img[6] / 10000.0                     # capa ya calculada

dif = np.abs(ndvi_mio - ndvi_tile)
print("Diferencia máxima entre mi NDVI y el del tile:", round(float(dif.max()), 4))
print("Coinciden — acabas de reproducir un producto satelital real.")

In [ ]:
# Mira el mapa de NDVI: verde = cultivo vigoroso, café = suelo, oscuro = agua
plt.figure(figsize=(7.5, 6))
im = plt.imshow(ndvi_mio, cmap="RdYlGn", vmin=0, vmax=0.9)
plt.title("NDVI — vigor del cultivo en el Valle del Yaqui")
plt.axis("off"); plt.colorbar(im, shrink=0.8, label="NDVI"); plt.show()

## Toda la familia, lado a lado

Cada índice resalta algo distinto. Verlos juntos muestra por qué un
clasificador se beneficia de las 13 capas: una parcela de trigo y una de
garbanzo podrían verse similares en NDVI pero diferir en un índice de agua o
de residuo.


In [ ]:
# Las capas 6..12 son los 7 índices, en este orden:
nombres_indice = ["NDVI", "EVI", "GCVI", "MSAVI2", "LSWI", "NDSVI", "NDTI"]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for k, ax in enumerate(axes.ravel()):
    if k < len(nombres_indice):
        capa = img[6 + k] / 10000.0
        im = ax.imshow(capa, cmap="RdYlGn")
        ax.set_title(nombres_indice[k]); ax.axis("off")
    else:
        ax.axis("off")
plt.suptitle("6 bandas espectrales se vuelven 7 índices = 13 pistas por píxel")
plt.tight_layout(); plt.show()

## Fenología: por qué un mes es una foto y una temporada es una historia

El NDVI en una sola fecha es una foto. Sigue la misma parcela *a lo largo*
de la temporada — siembra, verdeo, pico, cosecha — y su NDVI traza una curva
llamada su **fenología**. Esa huella temporal suele ser lo que separa dos
cultivos que se ven idénticos cualquier día suelto. Tu tile es un mes (marzo
2018); el pipeline de producción del Módulo 9 apila *muchos* meses justo
para capturar esto.


## 🧪 Ponte a prueba

**El NDVI es `(NIR − Rojo) / (NIR + Rojo)`. ¿Por qué dividir, en vez de solo
usar `NIR − Rojo`?**

<details><summary>Ver respuesta</summary>

Dividir lo vuelve un *cociente*, que cancela las diferencias de brillo
general (ángulo solar, pendiente del terreno, bruma ligera). `NIR − Rojo`
solo cambiaría con la iluminación aun para la misma planta sana; el cociente
normalizado lee el vigor de forma consistente, de −1 a +1.

</details>

**¿Para qué cargar EVI, LSWI, NDTI... si el NDVI ya mide el verdor?**

<details><summary>Ver respuesta</summary>

El NDVI se satura en doseles densos e ignora el agua y el residuo del suelo.
Otros índices capturan clorofila, agua del dosel y labranza/residuo. Dos
cultivos pueden compartir un NDVI pero diferir en estos — así que más
índices le dan al clasificador más formas de distinguirlos.

</details>


## 🔭 Profundiza

Opcional: estas tarjetas bilingües de conceptos amplían lo que acabas
de aprender (prerrequisitos, linaje a fundamentos, referencias):

- [NDVI, a fondo](https://abxda.github.io/rs-learning-audio/?id=ndvi&lang=es)
- [Índices de vegetación (la familia)](https://abxda.github.io/rs-learning-audio/?id=vegetation-indices&lang=es)
- [Índices espectrales](https://abxda.github.io/rs-learning-audio/?id=spectral-indices&lang=es)
- [Infrarrojo cercano (NIR)](https://abxda.github.io/rs-learning-audio/?id=near-infrared&lang=es)
- [Fenología (el calendario del cultivo)](https://abxda.github.io/rs-learning-audio/?id=phenology&lang=es)
- [Índice de Área Foliar](https://abxda.github.io/rs-learning-audio/?id=leaf-area-index&lang=es)



---

[← Anterior · Módulo 3 — Datos limpios: de las nubes a la geomediana](03_datos_limpios_geomediana.ipynb) · [Siguiente → · Módulo 5 — De píxeles a parcelas: segmentación](05_de_pixeles_a_parcelas.ipynb)
